In [ ]:
import json

import pandas as pd
import great_expectations as gx
import synapseclient

from agoradatatools.gx import GreatExpectationsRunner

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_nested_field_values_mostly_regex import (
    ExpectColumnNestedObjectRegexRule,
)
from expectations.expect_column_values_to_have_list_members_of_type import (
    ExpectColumnValuesToHaveListMembersOfType,
)
from expectations.expect_column_values_to_have_list_length_in_range import (
    ExpectColumnValuesToHaveListLengthInRange,
)

# Create Expectation Suite for Model Details Data

## Get Example Data File

In [ ]:
syn = synapseclient.Synapse()
syn.login()

In [ ]:
# Alternatively, use the local staging file: model_details_file = "../staging/model_details.json"
model_details_file = syn.get("syn73774519").path

## Create Validator Object on Data File

In [ ]:
df = pd.read_json(model_details_file, dtype={"jax_id": str})
nested_columns = ["genetic_info", "biomarkers", "pathology"]
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, nested_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "model_details"

## Add Expectations to Validator Object For Each Column

In [ ]:
# name
validator.expect_column_values_to_be_of_type("name", "str")
validator.expect_column_values_to_not_be_null("name")
validator.expect_column_values_to_be_unique("name")

In [ ]:
# model_type
validator.expect_column_values_to_be_of_type("model_type", "str")
validator.expect_column_values_to_not_be_null("model_type")
validator.expect_column_values_to_be_in_set("model_type", {"Familial AD", "Late Onset AD"})

In [ ]:
# contributing_group
validator.expect_column_values_to_be_of_type("contributing_group", "str")
validator.expect_column_values_to_not_be_null("contributing_group")
validator.expect_column_values_to_be_in_set("contributing_group", {"IU/Jax/Pitt", "UCI"})

In [ ]:
# study_synid
validator.expect_column_values_to_be_of_type("study_synid", "str")
validator.expect_column_values_to_not_be_null("study_synid")
validator.expect_column_values_to_match_regex("study_synid", regex="^syn\\d+$")

In [ ]:
# rrid
validator.expect_column_values_to_be_of_type("rrid", "str")
validator.expect_column_values_to_not_be_null("rrid")

In [ ]:
# jax_id (zero-padded 6-digit string; loaded with dtype={"jax_id": str} to preserve leading zeros)
validator.expect_column_values_to_be_of_type("jax_id", "str")
validator.expect_column_values_to_not_be_null("jax_id")
validator.expect_column_values_to_match_regex("jax_id", regex=r"^\d{6}$")

In [ ]:
# alzforum_id (empty string allowed for some models)
validator.expect_column_values_to_be_of_type("alzforum_id", "str")

In [ ]:
# genotype
validator.expect_column_values_to_be_of_type("genotype", "str")
validator.expect_column_values_to_not_be_null("genotype")

In [ ]:
# matched_controls
validator.expect_column_values_to_be_of_type("matched_controls", "list")
validator.expect_column_values_to_not_be_null("matched_controls")
validator.expect_column_values_to_have_list_length_in_range(column="matched_controls", list_length_range=[1, 10])
validator.expect_column_values_to_have_list_members_of_type(column="matched_controls", member_type="str")

In [ ]:
# aliases
validator.expect_column_values_to_be_of_type("aliases", "list")
validator.expect_column_values_to_not_be_null("aliases")
validator.expect_column_values_to_have_list_members_of_type(column="aliases", member_type="str")
validator.expect_column_values_to_have_list_length_in_range(column="aliases", list_length_range=[1, 5])

In [ ]:
# transcriptomics (nullable; when present, must be a comparison/expression URL path)
validator.expect_column_values_to_match_regex(
    "transcriptomics",
    regex="^comparison/expression",
    row_condition="transcriptomics.notnull()",
    condition_parser="pandas",
)

In [ ]:
# disease_correlation (nullable; when present, must be a comparison/correlation URL path)
validator.expect_column_values_to_match_regex(
    "disease_correlation",
    regex="^comparison/correlation",
    row_condition="disease_correlation.notnull()",
    condition_parser="pandas",
)

In [ ]:
# spatial_transcriptomics (always null in current data)
validator.expect_column_values_to_be_null("spatial_transcriptomics")

In [ ]:
# genetic_info (serialized JSON array of allele objects)
with open("../src/agoradatatools/great_expectations/gx/json_schemas/model_details/genetic_info_schema.json") as f:
    genetic_info_schema = json.load(f)

validator.expect_column_values_to_be_of_type("genetic_info", "str")
validator.expect_column_values_to_not_be_null("genetic_info")
validator.expect_column_values_to_match_json_schema("genetic_info", json_schema=genetic_info_schema)
validator.expect_column_nested_object_regex_rule(
    column="genetic_info",
    target_field="ensembl_gene_id",
    regex_pattern="^ENS(MUS)?G[0-9]{11}$"
)

In [ ]:
# biomarkers (serialized JSON array; can be empty for models without biomarker data)
# This field is serialized as a string for JSON schema validation.
with open("../src/agoradatatools/great_expectations/gx/json_schemas/model_details/biomarkers_schema.json") as f:
    biomarkers_schema = json.load(f)

validator.expect_column_values_to_be_of_type("biomarkers", "str")
validator.expect_column_values_to_not_be_null("biomarkers")
validator.expect_column_values_to_match_json_schema("biomarkers", json_schema=biomarkers_schema)

In [ ]:
# pathology (serialized JSON array; can be empty for models without pathology data)
# This field is serialized as a string for JSON schema validation.
validator.expect_column_values_to_be_of_type("pathology", "str")
validator.expect_column_values_to_not_be_null("pathology")

## Save Expectation Suite

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [ ]:
context.build_data_docs()
context.open_data_docs()